In [ ]:
!git clone https://github.com/weathon/vsf.git
!cd vsf && git stash && git switch fix_flux

Cloning into 'vsf'...
remote: Enumerating objects: 5103, done.
remote: Counting objects: 100% (329/329), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 5103 (delta 248), reused 159 (delta 152), pack-reused 4774 (from 2)
Receiving objects: 100% (5103/5103), 1.30 GiB | 49.53 MiB/s, done.
Resolving deltas: 100% (1899/1899), done.
Updating files: 100% (2108/2108), done.
No local changes to save
Branch 'fix_flux' set up to track remote branch 'fix_flux' from 'origin'.
Switched to a new branch 'fix_flux'


In [ ]:
!pip install compel

In [ ]:
import sys
import torch
sys.path.append("vsf")

import torch
import sys
from diffusers import FluxTransformer2DModel
model = FluxTransformer2DModel.from_single_file(
    "https://huggingface.co/xzyhku/flux_hpsv2.1_dancegrpo/blob/main/checkpoint-ema-300-0/diffusion_pytorch_model.safetensors",
    config="config.json",
    torch_dtype=torch.bfloat16,
)

from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("weathon/anti_aesthetics_dataset", split="train")

# from diffusers import FluxPipeline
from src.flux_pipeline import VSFFluxPipeline

dance_pipe = VSFFluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    torch_dtype=torch.bfloat16,
    transformer=model,
)
dance_pipe = dance_pipe.to("cuda")
import wandb
wandb.init(project="vsf_generation")
gen = []
for sample in ds:
    image = dance_pipe(
        sample["disorted_long_prompt"],
        negative_prompt=sample["negative_prompt"],
        guidance_scale=0.0,
        num_inference_steps=32,
        max_sequence_length=256,
        scale=3.0,
        generator=torch.Generator("cpu").manual_seed(54324)
    ).images[0]
    gen.append({
        "prompt": sample["disorted_long_prompt"],
        "negative_prompt": sample["negative_prompt"],
        "image": image,
        "method": "DanceFlux+VSF Strong"
    })
    wandb.log({
        "generated_images": wandb.Image(image, caption=sample["disorted_long_prompt"])
    })

from datasets import Dataset
gen_ds = Dataset.from_list(gen)
gen_ds.push_to_hub("weathon/vsf_aa_dance_3.0", private=True)


checkpoint-ema-300-0/diffusion_pytorch_m(…):   0%|          | 0.00/47.6G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/442 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/66.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/300 [00:00<?, ? examples/s]

model_index.json:   0%|          | 0.00/536 [00:00<?, ?B/s]

...

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   0%|          | 1.05MB /  398MB            

CommitInfo(commit_url='https://huggingface.co/datasets/weathon/vsf_aa_dance_3.0/commit/21a615a6ddb1dd55fe117b604622605555945c29', commit_message='Upload dataset', commit_description='', oid='21a615a6ddb1dd55fe117b604622605555945c29', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/weathon/vsf_aa_dance_3.0', endpoint='https://huggingface.co', repo_type='dataset', repo_id='weathon/vsf_aa_dance_3.0'), pr_revision=None, pr_num=None)